<a href="https://colab.research.google.com/github/AsiimweJr/LEARNING-Python_languages/blob/main/assignment1_individual_TB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Individual Assignment 1: Who is at risk of an unsuccessful TB treatment outcome?

**Fundamentals of Health Data Science**

### The situation

You work with the TB clinic of a district hospital. When a patient **starts** TB treatment, the clinic records some information about them. Six to eight months later, the treatment ends in one of several ways. Sadly, some patients **die**, are **lost to follow-up** (stop coming) or their treatment **fails**. Together these are called an **unsuccessful treatment outcome**.

The TB focal person can give extra support (home visits, phone follow-up, help with transport) to a limited number of patients. She asks:

> *"On the day a patient starts treatment, can we tell who is most at risk of an unsuccessful outcome, so we support them from the start?"*

The answer is **yes or no** (unsuccessful / successful), so this is a **classification** problem. You will build a **logistic regression** model, following the same journey as the demonstration notebook, but **you write all the code and make all the decisions.**

### Your data is personal

Everyone receives the same file, `tb_clinic_register.csv` (about 6,000 patients). At the start, you type your **registration number**. The notebook then picks **your own sample of 2,500 patients** and your own shuffle. Your numbers will differ from your classmates': **two students cannot have the same results.**

> The data are **simulated for teaching**. No real patients are in this file.

### Before you start: read this carefully

**This is an individual assignment.**
- Your code, your decisions and your written answers must be **your own work**.
- All written answers (*What you decided* and the ✍️ reflection questions) must be **in your own words. Do not use AI tools to write them.**
- **Any use of AI tools must be stated** in the declaration at the end: what you used it for, and where.
- If your submission suggests you may not understand your own work, you may be asked to complete a **short timed test or oral check** on your notebook, to confirm the work is yours.
- At the end you will declare that you have read through all your code, that it represents what you meant to do, and that you can **explain and defend every decision and every line**.

**About the reflection questions (✍️).**
They are **part of your mark**. They have **no single correct answer**: you are marked on how clearly you explain and defend **your own** decisions using **your own** results. They are **not** past or future exam questions, and you should not expect to see them in the exam.

**How this notebook works.**
- Cells marked **PROVIDED** are written for you: run them, do not edit them.
- Every other code cell says `# Write your code here`. You write the code. The demonstration notebook is your guide.
- Each step uses the same boxes:

| Box | What it is for |
|---|---|
| 📍 **Where you are** | where the previous step left you |
| 📘 **Concept** | the idea behind the step, in plain words |
| 🔀 **Your options** | choices you can try. None is compulsory; "leave it alone" is a valid choice if your results support it |
| 🔎 **Read further** | an official source and a plain-language source, if you want to go deeper |
| 🔮 **Predict first** | write what you expect *before* you run the code |
| 📝 **Record** | the options you tried and their scores |
| 🧭 **What you decided** | your decision, with your own numbers |
| ✅ **Checkpoint** | a provided check that tells you if a mechanical step went wrong (it never judges your decisions) |
| ✍️ **Reflection** | credited questions, in your own words |

**Deadline:** Friday 6 November 2026. **How it is marked:** details will be shared separately.

**What you hand in:** this notebook, with every cell run from top to bottom (*Restart and run all*), and the declaration completed.

### The columns in the file

| Column | Meaning |
|---|---|
| `patient_id` | clinic number: a label, not a measurement |
| `age_years` | age when treatment started |
| `sex` | recorded sex |
| `occupation` | farmer, fisherfolk, trader, salaried, boda boda rider |
| `subcounty` | where the patient lives (60 subcounties) |
| `hiv_positive` | 1 = living with HIV, 0 = HIV negative |
| `previous_tb_treatment` | 1 = has been treated for TB before |
| `distance_km` | distance from home to the clinic |
| `travel_cost_ugx` | return transport cost the patient reports (UGX) |
| `treatment_supporter` | 1 = has a named treatment supporter at home, 0 = none |
| `symptom_weeks_before_diagnosis` | how many weeks the patient had symptoms before being diagnosed |
| `weight_kg` | weight at the start of treatment |
| `sputum_smear_grade` | laboratory smear result: negative, scanty, 1+, 2+, 3+ |
| `alt_u_per_l`, `creatinine_umol_l`, `platelets_10e9_l`, `sodium_mmol_l` | routine blood tests at the start of treatment |
| `home_tracing_visits` | number of home visits made by health workers to trace the patient |
| **`unsuccessful_outcome`** | **what you predict**: 1 = died, lost to follow-up or treatment failed; 0 = successful |

---
## Step 1. Get your personal data (PROVIDED)

📍 **Where you are.** Nothing loaded yet.

Type your registration number between the quotes in `REG_NO`, **exactly as it appears on the class list** (same slashes, same letters), then run the cell. A different spelling gives a different sample. Keep `tb_clinic_register.csv` in the same folder as this notebook (on Google Colab, the cell asks you to upload it).

Write down the **data fingerprint** it prints: you will need it in your declaration.

In [1]:
# ===================== PROVIDED CELL: run it, do not edit it =====================
import os, hashlib
import numpy as np
import pandas as pd

REG_NO = ""   # <-- type your registration number here, e.g. "2025/HD07/1234U"

DATA_FILE = "tb_clinic_register.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files          # only on Google Colab
        print("Please upload tb_clinic_register.csv"); files.upload()
    except ImportError:
        raise FileNotFoundError("Put tb_clinic_register.csv in the same folder as this notebook.")

reg = REG_NO.strip().upper().replace(" ", "")
if reg == "":
    raise ValueError("Type your registration number in REG_NO (between the quotes), then run this cell again.")

MY_SEED = int(hashlib.sha256(reg.encode()).hexdigest(), 16) % (2**31)   # your personal seed
SPLIT_SEED = MY_SEED % 10000                                              # use this as random_state

everyone = pd.read_csv(DATA_FILE)
all_ids = np.sort(everyone["patient_id"].unique())
my_ids = np.random.default_rng(MY_SEED).choice(all_ids, 2500, replace=False)
my_data = everyone[everyone["patient_id"].isin(my_ids)].sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

fp = hashlib.sha256(",".join(sorted(my_ids)).encode()).hexdigest()[:8].upper()
FINGERPRINT = fp[:4] + "-" + fp[4:]

print(f"Registration number : {reg}")
print(f"Data fingerprint    : {FINGERPRINT}   <- write this in your declaration")
print(f"Your SPLIT_SEED     : {SPLIT_SEED}   <- use random_state=SPLIT_SEED whenever you shuffle or split")
print(f"Your data           : {my_data.shape[0]} rows, {my_data.shape[1]} columns, in a table called my_data")

ValueError: Type your registration number in REG_NO (between the quotes), then run this cell again.

✅ **Checkpoints (PROVIDED).** Run this cell once. It creates three small checks you will use later: `check_cleaning(...)`, `check_leakage(...)` and `check_split(...)`. They only look for mechanical mistakes. They never judge your decisions.

In [ ]:
# ===================== PROVIDED CELL: run it, do not edit it =====================
def _say(ok, good, bad):
    print(("  ✓ " + good) if ok else ("  ✗ " + bad))

def check_cleaning(df):
    """Checks that the obvious data problems have been dealt with. Pass your cleaned table."""
    print("Cleaning checkpoint")
    _say(not df["patient_id"].duplicated().any(), "one row per patient",
         "some patients still appear more than once")
    sexes = set(df["sex"].dropna().unique())
    _say(sexes <= {"F", "M"}, "sex is written one way", f"sex is still written in several ways: {sorted(sexes)}")
    _say(not (df["distance_km"] <= 0).any(), "no impossible distances", "some distances are 0 km or less")
    _say(not (df["travel_cost_ugx"] > 1_000_000).any(), "no impossible travel costs",
         "some travel costs are far too large to be real fares")
    _say(not ((df["weight_kg"] > 150) | (df["weight_kg"] < 25)).any(), "no impossible weights",
         "some weights are impossible for an adult")
    _say(not (df["symptom_weeks_before_diagnosis"] < 0).any(), "no negative symptom durations",
         "some symptom durations are negative (look at how 'not asked' was recorded)")
    print(f"  (for information) rows: {len(df)}, gaps per column: {df.isna().sum()[df.isna().sum() > 0].to_dict()}")

def check_leakage(df):
    """Checks that no column known only AFTER the outcome is still in your table."""
    print("Leakage checkpoint")
    _say("home_tracing_visits" not in df.columns, "no column known only after the outcome",
         "your table still contains a column that is only known after the outcome; go back to Step 4")
    if "patient_id" in df.columns:
        print("  (note) patient_id is still in the table: it is a label, not a clue. Do not use it as a predictor.")

def check_split(train, test, target="unsuccessful_outcome"):
    """Checks the train/test split. Pass your training table and your test table."""
    print("Split checkpoint")
    share = len(test) / (len(train) + len(test))
    _say(0.15 <= share <= 0.35, f"test set is {share:.0%} of patients", f"test set is {share:.0%} of patients; aim for 20-30%")
    _say(len(set(train.index) & set(test.index)) == 0, "no patient is in both sets", "some rows are in both sets")
    gap = abs(train[target].mean() - test[target].mean())
    _say(gap < 0.015, f"outcome rate similar in both sets ({train[target].mean():.1%} vs {test[target].mean():.1%})",
         f"outcome rate differs ({train[target].mean():.1%} vs {test[target].mean():.1%}); did you stratify?")

print("Checkpoints ready: check_cleaning, check_leakage, check_split")

---
## Step 2. First look

📍 **Where you are.** Your 2,500 patients are in `my_data`.

📘 **Concept.** Before anything clever, find out what you have: how many patients, what kind of columns, **how common the outcome is**, where the **gaps** are, and which values are **impossible**. A clinic export is never clean.

🧰 **Useful tools:** `.shape`, `.head()`, `.info()`, `.describe()`, `.isna().sum()`, `.value_counts()`, `.duplicated()`.

🔮 **Predict first.** What share of patients do you expect to have an unsuccessful outcome? Why does it matter?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Record what you found**

| Problem | How many rows | What you will do about it |
|---|---|---|
| | | |
| | | |
| | | |

---
## Step 3. Clean the data with written rules

📍 **Where you are.** A list of problems from Step 2.

📘 **Concept: cleaning rules.** Each rule should be **written down**, so anyone can repeat it, and **justified**: why is this value impossible? A useful habit is to **check one column against another**: a fare of 2 million UGX for someone living 3 km away is a typing error, but a large fare for someone living 120 km away may be real. Marking an impossible value as **missing** (`np.nan`) keeps the rest of the patient's information; deleting the whole row throws it away.
*Trap:* cleaning rules that *learn from the data* (for example "fill gaps with the average") must wait until after the split (Step 5). Fixed rules ("a weight of 540 kg is impossible") are safe now.

🔀 **Your options for each problem:** mark as missing · correct it (only if you are sure of the true value) · remove the row · keep it (if it is real)

🔎 **Read further:** [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)

In [ ]:
# Write your code here
# (then run the checkpoint below)

In [ ]:
# PROVIDED: if you named your cleaned table differently, change df to its name
try:
    check_cleaning(df)
except NameError:
    print("Nothing to check yet: write your cleaning code above first (call your cleaned table df).")

🧭 **What you decided** *(your rules, why each value was impossible, rows lost)*

**Your cleaning rules and reasons**

*Your answer:*

---
## Step 4. Check for leakage

📍 **Where you are.** A clean table.

📘 **Concept: leakage.** A column is **leaky** if it is only known **after** the outcome, or because of it. A model that uses it looks brilliant and is useless in practice, like a student who has seen the exam answers. For every column, ask:

> *"On the day the patient starts treatment, would the clinic already know this value?"*

*Trap:* a single column that predicts **almost perfectly** is far more likely to be leakage than a discovery.

🔀 **How to check:** read the column definitions · compare each column with the outcome (`pd.crosstab`, `groupby`) · score a model that uses one suspicious column alone

🔎 **Read further:** [scikit-learn: Common pitfalls (data leakage)](https://scikit-learn.org/stable/common_pitfalls.html)

🔮 **Predict first.** Which column(s) do you suspect, and why?
*Your prediction:*

In [ ]:
# Write your code here

In [ ]:
# PROVIDED: use the name of your table
try:
    check_leakage(df)
except NameError:
    print("Nothing to check yet: write your code above first.")

**✍️ Reflection 1 (credited, your own words).** The clinic manager is delighted: one column predicts outcomes almost perfectly. Explain to her, using **your own results**, why you removed it.

*Your answer:*

---
## Step 5. Split: practice patients and a sealed exam

📍 **Where you are.** A clean table without leaky columns.

📘 **Concept: the train/test split.** A model always looks good on the patients it learned from. The honest test uses patients it has **never seen**. So lock some patients away **now**, before you explore patterns or build anything.
- Use about **25%** for the test set.
- Use `stratify=` so the share of unsuccessful outcomes is the same in both sets (important when the outcome is rare).
- Use `random_state=SPLIT_SEED` (your personal seed).

**From here on, explore and decide using the training patients only. Do not look at the test set until Step 17.**

🔎 **Read further:** [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) · [Google ML Crash Course: Datasets, generalization and overfitting](https://developers.google.com/machine-learning/crash-course/overfitting)

In [ ]:
# Write your code here

In [ ]:
# PROVIDED: use the names of your training and test tables
try:
    check_split(train, test)
except NameError:
    print("Nothing to check yet: write your split code above first (call the tables train and test).")

---
## Step 6. Explore your training patients

📍 **Where you are.** Training patients only. The test set is sealed.

📘 **Concept: exploration guides the model.** A plain logistic regression assumes every clue pushes the risk up or down **in a straight line**, and that each clue acts **on its own**. Exploration shows you where those assumptions are wrong, and so which steps you will need.

Answer each question with a table or a picture (`groupby(...).mean()`, `pd.cut`, `.hist()`, `.plot.bar()`, `.corr()`):

1. **Lopsided numbers:** which columns have a long tail to the right?
2. **Curved risk:** does the risk of an unsuccessful outcome go up steadily with age, or does it bend? (Try age bands.)
3. **Clues that work together:** choose at least two pairs of clues that you think might act *together* (for example, something about **getting to the clinic** and something about **support at home**). Compare outcome rates across the combinations.
4. **Word columns:** do occupation and smear grade matter? Is there a natural *order* in either?
5. **Overlap:** do any two columns tell the same story?
6. **The blood tests:** do they show any pattern with the outcome?

🔮 **Predict first.** Before you look: which three clues do you expect to matter most?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Your to-do list from exploring**

| What you saw | What you will try | Step |
|---|---|---|
| | | |
| | | |
| | | |

---
## Step 7. A first, simple model: the baseline

📍 **Where you are.** A to-do list, no model yet.

📘 **Concept: baseline, cross-validation and pipelines.**
- **Baseline:** the simplest reasonable model (number columns as they arrived). Every later change must beat it.
- **AUC:** pick one patient with an unsuccessful outcome and one with a successful outcome. How often does the model give the first a higher risk? 0.5 = coin toss, 1.0 = perfect.
- **Cross-validation (CV):** five "mock exams" on the training patients. Report the **average and the spread**.
- **Pipeline:** put gap-filling, scaling, encoding and the model in **one** `Pipeline`, so each mock exam repeats the preparation from scratch and nothing leaks.

*Trap:* filling gaps or scaling **before** cross-validation, using all training patients at once. The checking group then helps prepare the data.

🔎 **Read further:** [scikit-learn: Pipelines and ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html) · [scikit-learn: Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html) · [scikit-learn: Logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) · [Google ML Crash Course: Logistic regression](https://developers.google.com/machine-learning/crash-course/logistic-regression)

🔮 **Predict first.** What AUC do you expect from the baseline?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Record:** baseline columns used: ___ · CV AUC: ___ ± ___

---
## Step 8. Transform single columns

📍 **Where you are.** A baseline score to beat.

📘 **Concept: transformations.** A transformation changes the **ruler** a column is measured on, not the patients.
- **Lopsided columns:** a log stretches small values and squeezes large ones (each mark means "twice as much", not "one unit more").
- **Curved risk:** if risk is high at both ends (a U-shape), a straight line cannot draw it, but adding a squared column lets the model bend.

*Analogy:* weighing a baby and an adult on the same bathroom scale. The scale works, but small differences in the baby disappear.
*Trap:* transforming because you can, not because your exploration gave you a reason.

🔀 **Your options for lopsided columns** *(distance, travel cost, symptom weeks)*

| Option | When it may help | How |
|---|---|---|
| Log | long right tail | `np.log(x)` (or `np.log1p(x)` if x can be 0) |
| Square root | moderate tail, counts | `np.sqrt(x)` |
| Bands | a threshold effect; easy to explain | `pd.cut(x, bins=[...])` then one-hot |
| Square | (rarely sensible for a right tail) | `x ** 2` |
| Leave it alone | the straight line already fits | (nothing) |

🔀 **Your options for a curved effect** *(age)*

| Option | How |
|---|---|
| Add a squared term, centred | `(age - centre) ** 2` (choose a sensible centre) |
| Add a squared term, not centred | `age ** 2` |
| Age bands | `pd.cut(age, bins=[...])` then one-hot |
| Distance from a centre | `abs(age - centre)` |
| Log of age | `np.log(age)` |
| Leave it alone | (nothing) |

> Tip: create your new columns with one function (for example `def make_features(data): ...`) and apply it to both training and test tables, so they are always built the same way.

🔎 **Read further:** [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) · [Google ML Crash Course: Numerical data](https://developers.google.com/machine-learning/crash-course/numerical-data)

🔮 **Predict first.** Which option do you expect to help most, and for which column?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Record**

| Option tried | Columns | CV AUC | Kept? |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

**🧭 What you decided**

*Your answer:*

**✍️ Reflection 2 (credited, your own words).** Which transformation(s) did you keep, and what in **your** exploration made you try them? If you left a column alone, explain why that was the right choice for your data.

*Your answer:*

---
## Step 9. Clues that work together: interactions

📍 **Where you are.** Transformed columns and a better score.

📘 **Concept: interactions.** Sometimes the effect of one clue **depends on another**. A plain model gives each clue **one** weight for everybody, so it averages the groups and gets both slightly wrong. An interaction column (clue A × clue B) lets the effect differ between groups.
*Clinical analogy:* a drug that only works in patients with a particular gene.
*Trap:* adding every possible pair. Most pairs are noise, and the model starts memorising (see Step 15).

🔀 **Candidates to test** *(or test your own, based on Step 6)*

| Candidate | How |
|---|---|
| distance × no treatment supporter | `log_distance * (1 - treatment_supporter)` |
| age × sex | `age_years * male` |
| HIV × weight | `hiv_positive * weight_kg` |
| previous TB treatment × smear grade | `previous_tb_treatment * smear_number` |
| no interaction | (nothing) |

🔎 **Read further:** [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)

🔮 **Predict first.** Which candidate, if any, will help? Why?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Record**

| Interaction tried | CV AUC | Kept? |
|---|---|---|
| | | |
| | | |

**✍️ Reflection 3 (credited, your own words).** Which interaction(s) did you keep? Using your results, explain why it would be a bad idea to add every possible pair of columns.

*Your answer:*

---
## Step 10. Turn words into numbers: encoding

📍 **Where you are.** Number columns are ready. Word columns are not yet used.

📘 **Concept: encoding.** A model multiplies each column by a weight and adds them up, so words must become numbers. How you do it matters:
- **Numbering** the categories (1, 2, 3...) tells the model they have an **order** and **equal steps**. That is right only if they really do.
- **One-hot** gives each category its own 0/1 column: no false order.
- **Grouping** joins categories that behave alike (based on knowledge, or on what you saw), giving fewer columns.

*Trap:* numbering categories that have no natural order.

🔀 **Your options** *(for `occupation`, `sputum_smear_grade` and `sex`; think about each one separately)*

| Option | How |
|---|---|
| One-hot | `OneHotEncoder` in the pipeline, or `pd.get_dummies` |
| Number in order | `.map({...})` with values in a meaningful order |
| Group, then one-hot | `.map({...})` to fewer groups |
| Two groups (0/1) | e.g. smear negative vs positive |
| Leave it out | (nothing) |

> `subcounty` has 60 levels. Think before you add it: what would 60 extra columns do with only a few hundred unsuccessful outcomes?

🔎 **Read further:** [scikit-learn: Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) · [Google ML Crash Course: Categorical data](https://developers.google.com/machine-learning/crash-course/categorical-data)

🔮 **Predict first.** Which column has a natural order, and which does not? What will that mean for numbering them?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Record**

| Column | Encoding tried | CV AUC | Kept? |
|---|---|---|---|
| | | | |
| | | | |

**✍️ Reflection 4 (credited, your own words).** Did you treat `occupation` and `sputum_smear_grade` the same way or differently? Explain why, using **what each column means** and **your results**.

*Your answer:*

---
## Step 11. Choose the final columns

📍 **Where you are.** Many candidate columns.

📘 **Concept: feature selection.** More columns are not automatically better. Each extra column can be wrong, missing, costly to collect, or a source of chance patterns. When two models score about the same, prefer the **simpler** or more **practical** one, and say why.

🔀 **Your options to consider**

| Option | Question to ask |
|---|---|
| Drop one of two overlapping columns | do they tell the same story? |
| Drop a column with many gaps | is it worth collecting? |
| Add sex | does it add anything once other clues are in? |
| Add the four blood tests | did Step 6 show any pattern? |
| Keep everything | is the extra complexity justified? |

🔎 **Read further:** [Google ML Crash Course: Datasets, generalization and overfitting](https://developers.google.com/machine-learning/crash-course/overfitting)

In [ ]:
# Write your code here

📝 **Record**

| Option | CV AUC | Kept? |
|---|---|---|
| | | |
| | | |

**✍️ Reflection 5 (credited, your own words).** Two of your models scored almost the same. Which did you choose and why? Think about the clinic (what is easy to collect, what has gaps), not only the number.

*Your answer:*

---
## Step 12. Your careful model, judged by cross-validation

📍 **Where you are.** A final list of columns and encodings.

Build your final pipeline and report its cross-validated AUC (average ± spread). Then draw your **journey**: one bar per step, showing how the score changed.

🔮 **Predict first.** How much did your decisions improve on the baseline?
*Your prediction:*

In [ ]:
# Write your code here

---
## Step 13. The rare outcome and the accuracy trap

📍 **Where you are.** A good cross-validated AUC.

📘 **Concept: imbalance.** When only about 1 patient in 8 has the outcome, a lazy rule ("everyone succeeds") already has high **accuracy** and finds **nobody**. Better scores here:
- **Recall (sensitivity):** of the patients who had an unsuccessful outcome, how many did the model flag?
- **Precision (PPV):** of the patients flagged, how many really had one?

*Trap:* judging a rare-outcome model by accuracy. Second trap: any resampling must happen **inside** the training folds only, never before the split.

🔀 **Your options**

| Option | How |
|---|---|
| Do nothing, cut-off 50% | `model.predict(...)` |
| Class weights | `LogisticRegression(class_weight="balanced")` |
| Lower the cut-off | use `predict_proba` and a smaller cut-off (Step 14) |
| Oversample the rare group | repeat rare patients in the training folds only (e.g. `imbalanced-learn`'s `RandomOverSampler` in an `imblearn` pipeline) |
| Undersample the common group | use fewer successful patients in the training folds only |

> Compare options on your **training** patients using fair, out-of-fold predictions: `cross_val_predict(model, train_f, train_f[Y], cv=cv, method="predict_proba")`. Keep the test set sealed.

🔎 **Read further:** [scikit-learn: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) · [Google ML Crash Course: ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)

🔮 **Predict first.** What will happen to recall and precision when you use class weights?
*Your prediction:*

In [ ]:
# Write your code here

📝 **Record**

| Option | Accuracy | Recall | Precision | Patients flagged |
|---|---|---|---|---|
| | | | | |
| | | | | |

**✍️ Reflection 6 (credited, your own words).** Explain to the TB focal person why accuracy is the wrong score for this question, using **your** numbers. Which score would you show her instead, and why?

*Your answer:*

---
## Step 14. Choose a cut-off for the TB focal person

📍 **Where you are.** A model that ranks patients, and a choice about imbalance.

📘 **Concept: the cut-off.** The model gives a probability. To act, you choose a cut-off: "support everyone above X%". This is a **practical** choice, not a statistical one. A lower cut-off finds more at-risk patients but needs more support visits.

> **Scenario:** the TB focal person can give extra support to about **1 new patient in 5**.

Choose your cut-off using your **training** patients (out-of-fold predictions), never the test set. A plot of recall, precision and "share flagged" against the cut-off helps.

🔎 **Read further:** [scikit-learn: Tuning the decision threshold](https://scikit-learn.org/stable/modules/classification_threshold.html)

In [ ]:
# Write your code here

**✍️ Reflection 7 (credited, your own words).** Where did you put your cut-off? What share of at-risk patients does it find, and who does it miss? Would you change it if the focal person could support twice as many patients?

*Your answer:*

---
## Step 15. Cramming, and tuning the strictness

📍 **Where you are.** A careful model.

📘 **Concept: overfitting (cramming).** A model with too many inputs for the number of patients memorises chance patterns in the training data. It scores **very well on the patients it learned from** and **worse on new patients**. The warning sign is a **big gap** between the two.

📘 **Concept: regularisation and tuning.** Logistic regression has a setting **C** that controls how strict the model is with itself (small C = strict, large C = relaxed). C is a **hyperparameter**: you choose it by testing with cross-validation (`GridSearchCV`), never on the test set.

**Your task:**
1. Build a deliberately over-complicated **kitchen-sink** model: every number column (including the four blood tests), occupation, sex and all 60 subcounties, plus **every pair** of number columns, with `C=10000`.
2. Compare its AUC on the **training patients it learned from** with its **cross-validated** AUC.
3. Tune C for the kitchen-sink model, and draw the training and CV scores against C.
4. Tune C (and, if you like, class weights) for **your careful model**. Does it need tuning?

🔎 **Read further:** [scikit-learn: Tuning hyperparameters](https://scikit-learn.org/stable/modules/grid_search.html) · [Google ML Crash Course: Datasets, generalization and overfitting](https://developers.google.com/machine-learning/crash-course/overfitting)

🔮 **Predict first.** Which will score higher on unseen patients: the kitchen sink or your careful model?
*Your prediction:*

In [ ]:
# Write your code here

**✍️ Reflection 8 (credited, your own words).** Describe what happened to the kitchen-sink model on unseen patients, and what tuning C did to it. Did your careful model need tuning? What does this tell you about when tuning matters?

*Your answer:*

---
## Step 16. The split lottery

📍 **Where you are.** Cross-validated scores for your choices.

📘 **Concept: one split is one lottery ticket.** With few patients who have the outcome, *which* patients land in the test set can move the score a lot. Repeat the whole split, train and test process with **30 different random seeds** (use `random_state=SPLIT_SEED + i`) on your cleaned data, and look at how much the test AUC varies.

🔎 **Read further:** [scikit-learn: Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html)

🔮 **Predict first.** How far apart will the lowest and highest test AUC be?
*Your prediction:*

In [ ]:
# Write your code here

🧭 **What you decided** *(how should model scores be reported, given your lottery?)*

*Your answer:*

---
## Step 17. Open the sealed exam, once

📍 **Where you are.** All decisions made on training patients: columns, encoding, imbalance approach, cut-off, C.

Now, **for the first and only time**, use the test set:
1. Fit your final model on all training patients.
2. Report on the test patients: AUC, and at **your** cut-off, the confusion matrix, recall and precision.
3. Draw four pictures: confusion matrix, ROC curve, precision-recall curve, calibration curve.

🔎 **Read further:** [scikit-learn: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) · [scikit-learn: Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)

In [ ]:
# Write your code here

🧭 **What you decided** *(did the test results match what cross-validation told you? Explain any difference.)*

*Your answer:*

---
## Step 18. Save your model and use it

📍 **Where you are.** A final model and a cut-off.

1. Save your model with `joblib`, together with your cut-off.
2. Write a short **model card** (5 to 8 lines): purpose, data, columns used, cross-validated and test scores, cut-off, limitations.
3. Load the saved model and predict for these three new patients. Which one would you support first?

| | age | sex | occupation | subcounty | HIV | previous TB | distance km | travel cost | supporter | symptom weeks | weight kg | smear | ALT | creatinine | platelets | sodium |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **A** | 19 | M | fisherfolk | subcounty_07 | 0 | 0 | 48 | 40000 | 0 | 14 | 51 | 2+ | 25 | 75 | 260 | 139 |
| **B** | 45 | F | salaried | subcounty_33 | 1 | 0 | 3 | 3000 | 1 | 3 | 62 | 1+ | 30 | 82 | 300 | 138 |
| **C** | 72 | F | farmer | subcounty_51 | 0 | 1 | 15 | 12000 | 1 | 8 | (not weighed) | negative | 28 | 90 | 250 | 136 |

> Hint: put the patients in a table with the same column names as the original file, apply the same feature steps, and use `predict_proba`.

In [ ]:
# Write your code here

**Your model card**

*Purpose:*
*Data:*
*Columns used:*
*Scores (cross-validated and test):*
*Cut-off and what it means:*
*Limitations:*

**✍️ Reflection 9 (credited, your own words).** Imagine this model is used in the clinic next year. Name two things that could make it stop working well, and say how the clinic would notice.

*Your answer:*

---
## Declaration (complete before you submit)

Type your answers after each item.

1. **Registration number:**
2. **Data fingerprint** (printed in Step 1):
3. I have run this notebook from top to bottom (*Restart and run all*) before submitting. **(yes / no):**
4. **I have read through all the code in this notebook, and it represents what I intended to do. (yes / no):**
5. **I can explain and defend every decision and every line of code in this notebook. (yes / no):**
6. **I wrote all written answers in my own words, without AI tools. (yes / no):**
7. **Use of AI tools:** I used AI tools for *(write "none", or describe exactly what you used them for, and in which steps)*:
8. I understand that I may be asked to complete a **short timed test or oral check** on this work. **(yes / no):**

**Name:**   **Date:**